# Cleaning3
## Inizializzazione ed Import

In [4]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Get the files 
Exclusevily from ADNI dataset stored in the Datalake

In [5]:
file_codes = ['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA']

# 'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UPENN_ROI_MARS'


### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'
# 'UCSDVOL', 'UPENN_ROI_MARS',  --> do NOT use FreeSurfer but other model or Atlas so not comparable

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [6]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

In [7]:
len(zip_files)

8

## Operazioni
- Trasformare i volumi come percentuali di ICV

In [8]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned2'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned3'

In [9]:
if os.path.isfile(new_name+'.xlsx'):
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)

The ADNI_variables_cleaned3 file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned3 file has restored the previous information of the file_code: ['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA']
Open the file and verify it, if needed update the variables names and metadata


In [10]:
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [11]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    #new_support_file = dataCleaner.update_self_support_file(new_support_file)
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(df_new, file_code)
        # Transform volumes as ICV percentage
        final_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    else:
        final_df = df_new

    if 'FSVERSION' in final_df.columns:
        print(final_df['FSVERSION'].unique())
    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_03', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # Create new file name for datalake
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_03')

   
    #upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 



 ---- ADNIMERGE_25Jul2025_02.csv
[4.3 nan 5.1 6. ]


In [13]:
zip_files.keys()

dict_keys(['ADNIMERGE_25Jul2025_02.csv', 'MMSE_25Jul2025_02.csv', 'ADSP_PHC_BIOMARKER_25Jul2025_02.csv', 'ADAS_28Oct2025_02.csv', 'FAQ_28Oct2025_02.csv', 'CDR_28Oct2025_02.csv', 'MOCA_28Oct2025_02.csv', 'ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_02.csv'])

In [9]:
final_df.columns

Index(['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'FLDSTRENG', 'IMAGEUID',
       'STATUS', 'FSVERSION', 'ICV%ICV', 'Fusiform%ICV', 'Ventricles%ICV',
       'Entorhinal%ICV', 'MidTemp%ICV', 'Hippocampus%ICV'],
      dtype='object')

In [10]:
updated_metadata

{'cofattori': [],
 'file_code': 'UCSFFSL',
 'level': 'cleaned_03',
 'norm_intervallo': [],
 'norm_scala': [],
 'norm_scale_value': [],
 'norm_volume': ['ICV%ICV',
  'Fusiform%ICV',
  'Ventricles%ICV',
  'Entorhinal%ICV',
  'MidTemp%ICV',
  'Hippocampus%ICV'],
 'population': ['ADNI1', 'ADNIGO', 'ADNI2'],
 'predittori': ['ICV%ICV',
  'Fusiform%ICV',
  'Ventricles%ICV',
  'Entorhinal%ICV',
  'MidTemp%ICV',
  'Hippocampus%ICV'],
 'source': 'ADNI',
 'volume_norm_values': {'Ventricles%ICV': [0.11, 7.3, 'increasing'],
  'Hippocampus%ICV': [0.22, 0.66, 'inverse'],
  'Entorhinal%ICV': [0.07, 0.39, 'inverse'],
  'Fusiform%ICV': [0.63, 1.62, 'inverse'],
  'MidTemp%ICV': [0.7, 1.8, 'inverse']}}